# Jon2 AI — gated Kaggle GPU training

This private kernel trains only the original Jon2 model. It requires a real CUDA GPU and a passing read-only preflight for exact source approvals, at least 1,000,000 corpus characters, and at least 20 approved tokens per parameter. It never substitutes an external model. If any gate fails, training stops before changing weights.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import torch

print("JON2 GPU PREFLIGHT | PyTorch:", torch.__version__)
print("JON2 GPU PREFLIGHT | CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "Jon2 training stopped: Kaggle did not allocate a CUDA GPU. "
        "No CPU fallback is allowed for this job."
    )

gpu = torch.cuda.get_device_properties(0)
print("JON2 GPU:", gpu.name)
print("JON2 GPU memory GiB:", round(gpu.total_memory / (1024 ** 3), 2))

input_root = Path("/kaggle/input")
manifests = sorted(input_root.glob("**/source_approvals.json"))
if not manifests:
    raise FileNotFoundError(
        "Private approved training dataset is not mounted. "
        "Attach jon2-approved-training-data-v2 to this Kaggle kernel."
    )

manifest_path = manifests[0]
data_dir = manifest_path.parent
engine_input = data_dir.parent
engine_work = Path("/kaggle/working/jon2-engine")
if engine_work.exists():
    shutil.rmtree(engine_work)
engine_work.mkdir(parents=True)

required_modules = (
    "jon2_core.py",
    "train_jon2.py",
    "tokenizer.py",
    "source_approval.py",
    "corpus_reader.py",
    "source_split.py",
    "model_profiles.py",
    "preflight_training.py",
)
for name in required_modules:
    source = engine_input / name
    if not source.is_file():
        raise FileNotFoundError(f"Training module missing from private dataset: {source}")
    shutil.copy2(source, engine_work / name)

env = dict(os.environ)
env["PYTHONPATH"] = str(engine_work)

preflight_report = Path("/kaggle/working/jon2-preflight.json")
preflight_command = [
    sys.executable,
    str(engine_work / "preflight_training.py"),
    "--data", str(data_dir),
    "--approval-manifest", str(manifest_path),
    "--profile", "arabic_30m",
    "--minimum-characters", "1000000",
    "--minimum-tokens-per-parameter", "20",
    "--output", str(preflight_report),
]
print("Running read-only rights/data/model preflight before training...")
preflight = subprocess.run(preflight_command, env=env, check=False)
if preflight.returncode != 0:
    if preflight_report.is_file():
        print(preflight_report.read_text(encoding="utf-8"))
    raise RuntimeError(
        "Jon2 training is blocked by the approved-data preflight. "
        "No weights were created or changed."
    )

checkpoint_dir = Path("/kaggle/working/jon2-checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)
best_path = checkpoint_dir / "jon2-arabic-30m.pt"
last_path = checkpoint_dir / "jon2-arabic-30m-last.pt"

train_command = [
    sys.executable,
    str(engine_work / "train_jon2.py"),
    "--profile", "arabic_30m",
    "--data", str(data_dir),
    "--approval-manifest", str(manifest_path),
    "--steps", "1000",
    "--minimum-chars", "1000000",
    "--minimum-tokens-per-parameter", "20",
    "--require-cuda",
    "--output", str(best_path),
    "--last-checkpoint", str(last_path),
    "--keep-last",
]
print("Starting the original Jon2 Arabic 30M training run on the allocated Kaggle GPU...")
subprocess.run(train_command, env=env, check=True)

for path in (best_path, last_path):
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Training output is missing or empty: {path}")

files = {}
for label, path in (("best", best_path), ("resume", last_path)):
    raw = path.read_bytes()
    files[label] = {
        "filename": path.name,
        "size_bytes": len(raw),
        "sha256": hashlib.sha256(raw).hexdigest(),
    }

manifest = {
    "format": "jon2-private-checkpoint-manifest-v1",
    "profile": "arabic_30m",
    "training_status": "completed_pending_heldout_and_human_review",
    "gpu": gpu.name,
    "pytorch": torch.__version__,
    "cuda": torch.version.cuda,
    "files": files,
    "public_release_approved": False,
    "human_review_required": True,
}
manifest_path_out = checkpoint_dir / "checkpoint-manifest.json"
manifest_path_out.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(json.dumps(manifest, ensure_ascii=False, indent=2))


## Release policy

A successful training run is not proof of Arabic understanding. Keep the checkpoint private and public chat disabled until the held-out Egyptian Arabic/MSA benchmark and human semantic review pass.
